# Reproduce Table 1 — AAS Results
Compares our macro-averaged results across 13 WildlifeReID-10k datasets
against the paper's claimed AAS row (Table 1).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from experiments.aggregate_results import aggregate, PAPER_TARGETS, METRICS

In [ ]:
RESULTS_DIR = '../experiments/results'
df = aggregate(RESULTS_DIR)
df

In [ ]:
# Display mean metrics as percentages
mean_cols = [f'{k}_mean' for k in METRICS if f'{k}_mean' in df.columns]
(df[mean_cols] * 100).round(2)

In [ ]:
# Comparison: our macro-average vs paper targets
if 'MACRO_AVG' in df.index:
    macro = df.loc['MACRO_AVG']
    comparison = pd.DataFrame({
        'Ours (%)':  {k: macro.get(f'{k}_mean', float('nan')) * 100 for k in PAPER_TARGETS},
        'Paper (%)': {k: v * 100 for k, v in PAPER_TARGETS.items()},
    })
    comparison['Delta (%)'] = comparison['Ours (%)'] - comparison['Paper (%)']
    print(comparison.round(2).to_string())
else:
    print('No results yet — run experiments/run_all.sh first.')

In [ ]:
# Bar chart: our mAP per dataset vs paper target
if 'MACRO_AVG' in df.index:
    per_dataset = df.drop('MACRO_AVG')
    datasets = per_dataset.index.tolist()
    our_map = per_dataset['mAP_mean'].values * 100

    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(datasets))
    ax.bar(x, our_map, width=0.6, label='Ours')
    ax.axhline(PAPER_TARGETS['mAP'] * 100, color='red', linestyle='--', label='Paper (macro avg)')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('mAP (%)')
    ax.set_title('AAS Reproduction: mAP per Dataset')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../experiments/results/table1_map.png', dpi=150)
    plt.show()